In [ ]:
import pandas as pd
import altair as alt
import os

df = pd.read_csv('results.csv')
df['resolution'] = df['screen_pct'].astype(str) + '%'

metrics = [
    ('avg_GPU/TAA',            'TAA GPU Cost (ms)'),
    ('avg_GPUTime',            'Total GPU Time (ms)'),
    ('avg_RenderThreadTime',   'Render Thread Time (ms)'),
    ('avg_GPUMem/LocalUsedMB', 'VRAM (MB)'),
]

taa_params = [
    'r.TemporalAACurrentFrameWeight',
    'r.TemporalAASamples',
    'r.TemporalAAFilterSize',
    'r.TemporalAA.HistoryScreenPercentage',
]

res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

os.makedirs('plots', exist_ok=True)

for scene in df['scene'].unique():
    scene_df = df[df['scene'] == scene]
    
    rows = []
    for metric_col, metric_label in metrics:
        cols = []
        for param in taa_params:
            subset = scene_df[scene_df['cvar_name'] == param].copy()

            if subset.empty:
                continue

            # short param label for column title
            short_param = param.split('r.')[-1]

            chart = alt.Chart(subset).mark_line(point=True).encode(
                x=alt.X('cvar_value:Q', title=short_param),
                y=alt.Y(f'{metric_col}:Q', title=metric_label),
                color=alt.Color('resolution:N', scale=color_scale, sort=res_order,
                                legend=alt.Legend(title='Resolution')),
                tooltip=['resolution', 'cvar_value', metric_col]
            ).properties(
                width=220,
                height=180
            )
            cols.append(chart)

        if cols:
            rows.append(alt.hconcat(*cols))

    if rows:
        full_grid = alt.vconcat(*rows).properties(
            title=alt.TitleParams(
                text=scene,
                fontSize=16,
                anchor='start'
            )
        ).resolve_scale(
            y='independent'  # each row has its own y scale
        )

        full_grid.save(f'plots/{scene}_grid.html')
        full_grid.display()

In [ ]:
## With parquet errors
import pandas as pd
import altair as alt
import os

# --- Configuration ---
df_results = pd.read_csv('results.csv')
PARQUET_DIR = 'raw' # Folder where your .parquet files live
df_results['resolution'] = df_results['screen_pct'].astype(str) + '%'

metrics = [
    ('GPU/TAA',            'TAA GPU Cost (ms)'),
    ('GPUTime',            'Total GPU Time (ms)'),
    ('RenderThreadTime',   'Render Thread Time (ms)'),
    ('GPUMem/LocalUsedMB', 'VRAM (MB)'),
]

taa_params = [
    'r.TemporalAACurrentFrameWeight',
    'r.TemporalAASamples',
    'r.TemporalAAFilterSize',
    'r.TemporalAA.HistoryScreenPercentage',
]

res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

# --- 1. Data Hydration (Loading Parquet stats) ---
expanded_rows = []

print("Loading parquet data for error bars...")
for idx, row in df_results.iterrows():
    parquet_name = row['csv_file'].replace('.csv', '.parquet')
    parquet_path = os.path.join(PARQUET_DIR, parquet_name)
    
    if os.path.exists(parquet_path):
        data = pd.read_parquet(parquet_path)
        # Apply the same 150-frame warm-up skip used in your generation script
        df_analysis = data.iloc[150:] 
        
        # Calculate stats for the current run
        stats = {
            'scene': row['scene'],
            'resolution': row['resolution'],
            'cvar_name': row['cvar_name'],
            'cvar_value': row['cvar_value']
        }
        
        for m_col, _ in metrics:
            if m_col in df_analysis.columns:
                stats[f'{m_col}_avg'] = df_analysis[m_col].mean()
                stats[f'{m_col}_std'] = df_analysis[m_col].std()
        
        expanded_rows.append(stats)

plot_df = pd.DataFrame(expanded_rows)

# --- 2. Plotting with Error Bars ---
os.makedirs('plots', exist_ok=True)

for scene in plot_df['scene'].unique():
    scene_df = plot_df[plot_df['scene'] == scene]
    rows = []

    for metric_col, metric_label in metrics:
        cols = []
        avg_col = f'{metric_col}_avg'
        std_col = f'{metric_col}_std'

        for param in taa_params:
            subset = scene_df[scene_df['cvar_name'] == param].copy()
            if subset.empty: continue

            # Calculate upper and lower bounds for the error bars
            subset['lower'] = subset[avg_col] - subset[std_col]
            subset['upper'] = subset[avg_col] + subset[std_col]

            base = alt.Chart(subset).encode(
                x=alt.X('cvar_value:Q', title=param.split('r.')[-1]),
                color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
            )

            # The line and points
            line = base.mark_line(point=True).encode(
                y=alt.Y(f'{avg_col}:Q', title=metric_label),
                tooltip=['resolution', 'cvar_value', avg_col, std_col]
            )

            # The error bars (Standard Deviation)
            error = base.mark_errorbar().encode(
                y=alt.Y('lower:Q', title=metric_label),
                y2='upper:Q'
            )

            chart = (error + line).properties(width=220, height=180)
            cols.append(chart)

        if cols:
            rows.append(alt.hconcat(*cols))

    if rows:
        full_grid = alt.vconcat(*rows).properties(
            title=alt.TitleParams(text=f"{scene} (±1 SD)", fontSize=16, anchor='start')
        ).resolve_scale(y='independent')

        full_grid.save(f'plots/{scene}_error_bars.html')
        full_grid.display()

In [ ]:
df = pd.read_csv('results.csv')
subset = df[df['cvar_name'] == 'r.TemporalAA.HistoryScreenPercentage']
print(subset[['scene', 'screen_pct', 'cvar_value', 'avg_GPUTime', 'avg_GPU/TAA']])

In [ ]:
df = pd.read_csv('results.csv')

# print all rows for one specific parameter to see if values actually vary
subset = df[df['cvar_name'] == 'r.TemporalAA.HistoryScreenPercentage']
print(subset[['scene', 'screen_pct', 'cvar_value', 'avg_GPUTime', 'avg_GPUMem/LocalUsedMB', 'frames_analysed']])

In [ ]:
import pandas as pd
import altair as alt
import os

df = pd.read_csv('results_median.csv')  # changed
df['resolution'] = df['screen_pct'].astype(str) + '%'

metrics = [
    ('median_GPU/TAA',            'TAA GPU Cost (ms)'),       # changed
    ('median_GPUTime',            'Total GPU Time (ms)'),      # changed
    ('median_RenderThreadTime',   'Render Thread Time (ms)'),  # changed
    ('median_GPUMem/LocalUsedMB', 'VRAM (MB)'),                # changed
]

# everything below this line is identical to your existing code
taa_params = [
    'r.TemporalAACurrentFrameWeight',
    'r.TemporalAASamples',
    'r.TemporalAAFilterSize',
    'r.TemporalAA.HistoryScreenPercentage',
]
res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

os.makedirs('plots', exist_ok=True)
for scene in df['scene'].unique():
    scene_df = df[df['scene'] == scene]
    rows = []
    for metric_col, metric_label in metrics:
        cols = []
        for param in taa_params:
            subset = scene_df[scene_df['cvar_name'] == param].copy()
            if subset.empty:
                continue
            short_param = param.split('r.')[-1]
            chart = alt.Chart(subset).mark_line(point=True).encode(
                x=alt.X('cvar_value:Q', title=short_param),
                y=alt.Y(f'{metric_col}:Q', title=metric_label),
                color=alt.Color('resolution:N', scale=color_scale, sort=res_order,
                                legend=alt.Legend(title='Resolution')),
                tooltip=['resolution', 'cvar_value', metric_col]
            ).properties(width=220, height=180)
            cols.append(chart)
        if cols:
            rows.append(alt.hconcat(*cols))
    if rows:
        full_grid = alt.vconcat(*rows).properties(
            title=alt.TitleParams(text=scene, fontSize=16, anchor='start')
        ).resolve_scale(y='independent')
        full_grid.save(f'plots/{scene}_grid.html')
        full_grid.display()

In [ ]:
import pandas as pd
import altair as alt
import os

# --- 1. Configuration ---
# List your files here; the script will loop through each one
files_to_plot = ['results-10reps.csv', 'results-4reps.csv']

metrics = [
    ('GPU/TAA',           'TAA GPU Cost (ms)'),
    ('GPUTime',           'Total GPU Time (ms)'),
    ('RenderThreadTime',  'Render Thread Time (ms)'),
    ('GPUMem/LocalUsedMB', 'VRAM (MB)'),
]

taa_params = [
    'r.TemporalAACurrentFrameWeight',
    'r.TemporalAASamples',
    'r.TemporalAAFilterSize',
    'r.TemporalAA.HistoryScreenPercentage',
]

res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

os.makedirs('plots', exist_ok=True)

# --- 2. The Main Loop (One set of plots per CSV) ---
for csv_name in files_to_plot:
    if not os.path.exists(csv_name):
        print(f"⚠️  Skipping {csv_name}: File not found.")
        continue

    df = pd.read_csv(csv_name)
    df['resolution'] = df['screen_pct'].astype(str) + '%'
    
    # Identify run type from filename for the title (e.g., "10reps")
    run_label = csv_name.replace('results-', '').replace('.csv', '')

    for scene in df['scene'].unique():
        scene_df = df[df['scene'] == scene]
        rows = []

        for metric_col, metric_label in metrics:
            cols = []
            avg_col = f'avg_{metric_col}'
            std_col = f'std_{metric_col}'

            for param in taa_params:
                subset = scene_df[scene_df['cvar_name'] == param].copy()
                if subset.empty: continue

                # Error bar bounds using your pre-saved Std Dev
                subset['lower'] = subset[avg_col] - subset[std_col]
                subset['upper'] = subset[avg_col] + subset[std_col]

                base = alt.Chart(subset).encode(
                    x=alt.X('cvar_value:Q', 
                        title=param.replace('r.', ''), 
                        scale=alt.Scale(zero=False) # <--- This prevents the axis from starting at 0
                    ),
                    color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
                )

                # The Line (Mean)
                line = base.mark_line(point=True).encode(
                    y=alt.Y(f'{avg_col}:Q', title=metric_label),
                    tooltip=['resolution', 'cvar_value', avg_col, std_col]
                )

                # The Error Bars (Standard Deviation)
                error = base.mark_errorbar().encode(
                    y=alt.Y('lower:Q', title=metric_label),
                    y2='upper:Q'
                )

                chart = (error + line).properties(width=240, height=190)
                cols.append(chart)

            if cols:
                rows.append(alt.hconcat(*cols))

        if rows:
            # Create a combined grid for this scene/file combo
            full_grid = alt.vconcat(*rows).properties(
                title=alt.TitleParams(
                    text=f"Scene: {scene} ({run_label})",
                    subtitle=["Error bars represent ±1 Standard Deviation"],
                    fontSize=18, anchor='start'
                )
            ).resolve_scale(y='independent')

            # Save with a unique filename
            output_filename = f'plots/{scene}_{run_label}.html'
            full_grid.save(output_filename)
            print(f"✅ Generated: {output_filename}")
            
            # Use display() to show them in the notebook if running interactively
            full_grid.display()